# 06 · Writing to Multiple Sinks

> **Module 2 · Spark Fundamentals · Lecture 2.6**

In this notebook you will:

- Write to GCS with every mode and control the layout with `partitionBy`
- Write to BigQuery with `writeDisposition`, `temporaryGcsBucket`, and `writeMethod`
- Write to JDBC and Pub/Sub through `foreachPartition` with batching
- Fan out one DataFrame to several sinks efficiently with `cache()`
- Design idempotent writes with partition overwrite, and apply a production sink checklist

---


## Setup / prerequisites

- **Python 3.9–3.11** and **Java 11 or 17** on your machine.
- **PySpark 3.5.x**: `pip install "pyspark==3.5.1" pandas pyarrow numpy`
- This notebook is **self-contained and offline**: every code cell runs top-to-bottom with
  no GCP account. All data operations use local paths under `/tmp/`.
- GCS / BigQuery / Pub/Sub / JDBC code appears in **clearly-marked optional cells** that
  you uncomment after authenticating with `gcloud auth application-default login`.
- On Dataproc, **do not `pip install pyspark`** — use the Spark runtime already on the
  image (Spark 3.5 on Dataproc 2.2).


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pyspark

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("notebook-01")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print("PySpark :", pyspark.__version__)
print("Spark   :", spark.version)
print("Master  :", spark.sparkContext.master)


In [ ]:
# ------------------------------------------------------------------
# Self-contained sample data for the sink exercises.
# ------------------------------------------------------------------
import os
import shutil

from pyspark.sql import functions as F

DATA_DIR = "/tmp/dataproc_notebook_06"
if os.path.exists(DATA_DIR):
    shutil.rmtree(DATA_DIR)
os.makedirs(DATA_DIR, exist_ok=True)

orders = spark.createDataFrame(
    [
        ("o-1", "c-1", "EMEA", 100.50, "2026-10-01"),
        ("o-2", "c-2", "APAC",  80.00, "2026-10-01"),
        ("o-3", "c-1", "EMEA",  49.50, "2026-10-02"),
        ("o-4", "c-3", "AMER", 129.00, "2026-10-02"),
        ("o-5", "c-2", "APAC",  19.99, "2026-10-03"),
    ],
    schema="order_id string, customer_id string, region string, amount double, order_date string",
)
orders.show(truncate=False)

## 1 · Write modes

| Mode | Behaviour | Use |
|---|---|---|
| `append` (default) | Add to existing data | Incremental loads |
| `overwrite` | Replace the target entirely | Full refresh |
| `errorifexists` | Fail if the target exists | **Safe default for pipelines** |
| `ignore` | Do nothing if the target exists | Idempotent re-runs |

In [ ]:
out = f"{DATA_DIR}/orders_parquet"

orders.write.mode("overwrite").parquet(out)
print("overwrite -> files:", len([f for f in os.listdir(out) if f.endswith(".parquet")]))

orders.write.mode("append").parquet(out)
print("append    -> files:", len([f for f in os.listdir(out) if f.endswith(".parquet")]))
print("rows now  :", spark.read.parquet(out).count())

# errorifexists protects you from a silent overwrite.
try:
    orders.write.mode("errorifexists").parquet(out)
except Exception as exc:
    print("errorifexists raised:", type(exc).__name__)

# ignore is a no-op when the target exists.
orders.write.mode("ignore").parquet(out)
print("ignore    -> rows still:", spark.read.parquet(out).count())

## 2 · `partitionBy`

`partitionBy` writes one directory per distinct value, and a matching filter on read
prunes whole directories.

**Caution:** **Never partition on a high-cardinality column.** `order_id` would create millions of
tiny directories — on GCS, millions of objects with millisecond latency each. Rule of
thumb: fewer than ~10,000 partitions per table and at least ~100 MB per partition.

Good partition columns: `order_date`, `region`, `event_type`.
Bad: `user_id`, `session_id`, `transaction_id`.

In [ ]:
partitioned_out = f"{DATA_DIR}/orders_partitioned"
orders.write.mode("overwrite").partitionBy("order_date").parquet(partitioned_out)

for root, _, files in sorted(os.walk(partitioned_out)):
    rel = root.replace(partitioned_out, "")
    parquet_files = [f for f in files if f.endswith(".parquet")]
    if parquet_files:
        print(f"{rel}: {len(parquet_files)} file(s)")

# Partition pruning on read.
pruned = spark.read.parquet(partitioned_out).filter(F.col("order_date") == "2026-10-02")
pruned.explain(mode="formatted")

## 3 · BigQuery writes

| Option | Meaning |
|---|---|
| `writeDisposition` | `WRITE_APPEND`, `WRITE_EMPTY`, `WRITE_TRUNCATE` |
| `createDisposition` | `CREATE_IF_NEEDED` (default), `CREATE_NEVER` |
| `temporaryGcsBucket` | **Required** staging bucket for the load job |
| `writeMethod` | `direct` (default) or `indirect` |
| `insertMethod` | `insert` (default) or `load` |

`writeMethod=direct` streams rows via the Storage Write API with no GCS staging.
`writeMethod=indirect` writes JSON files to `temporaryGcsBucket` and runs a BigQuery
load job — useful for very large writes or when you need the load-job semantics.

**Caution:** `temporaryGcsBucket` is **mandatory** for BigQuery writes from Spark.

### Optional: BigQuery write (requires a GCP account)

Uncomment after `gcloud auth application-default login`.

In [ ]:
# ------------------------------------------------------------------
# OPTIONAL — BigQuery write. Uncomment to run.
# ------------------------------------------------------------------
# BQ_PROJECT = "your-project-id"
# BQ_DATASET = "sales"
# BQ_TABLE = "orders_gold"
#
# (
#     orders.write
#     .format("bigquery")
#     .option("writeDisposition", "WRITE_APPEND")
#     .option("temporaryGcsBucket", "your-staging-bucket")
#     .option("writeMethod", "direct")
#     .save(f"{BQ_PROJECT}.{BQ_DATASET}.{BQ_TABLE}")
# )

print("BigQuery write cell is commented out — no GCP account required for this notebook.")

## 4 · JDBC writes via `foreachPartition`

The built-in JDBC writer opens a transaction per partition, which is slow. The
`foreachPartition` idiom batches rows per partition and commits once — far fewer
round trips.

**Caution:** The connection, statement, and batch objects must be created **inside** the function:
the closure is serialized and shipped to the executors, and a connection opened on the
driver is not usable there.

In [ ]:
# ------------------------------------------------------------------
# OPTIONAL — JDBC write via foreachPartition. Uncomment to run.
# ------------------------------------------------------------------
# def write_jdbc_partition(iterator):
#     import pyodbc
#
#     rows = []
#     for row in iterator:
#         rows.append((row["order_id"], row["customer_id"], row["region"],
#                      row["amount"], row["order_date"]))
#     if not rows:
#         return
#
#     conn = pyodbc.connect(
#         "DRIVER={PostgreSQL Unicode};"
#         "SERVER=db-host;PORT=5432;DATABASE=mydb;"
#         "UID=user;PWD=secret",
#         autocommit=False,
#     )
#     try:
#         cursor = conn.cursor()
#         cursor.fast_executemany = True
#         cursor.executemany(
#             "INSERT INTO public.orders "
#             "(order_id, customer_id, region, amount, order_date) VALUES (?, ?, ?, ?, ?)",
#             rows,
#         )
#         conn.commit()
#     finally:
#         conn.close()
#
# orders.foreachPartition(write_jdbc_partition)

# A local, runnable stand-in that proves the batching idiom works.
OUT_DIR_BATCHES = f"{DATA_DIR}/jdbc_style_batches"

def write_batches(iterator):
    # foreachPartition passes exactly ONE argument: the partition iterator.
    # Anything else the closure needs is captured from the enclosing scope.
    import json
    import os
    out_dir = OUT_DIR_BATCHES
    os.makedirs(out_dir, exist_ok=True)
    batch = []
    for row in iterator:
        batch.append(row.asDict())
        if len(batch) >= 2:
            with open(os.path.join(out_dir, f"batch-{len(batch)}.json"), "w", encoding="utf-8") as fh:
                json.dump(batch, fh, default=str)
            batch = []
    if batch:
        with open(os.path.join(out_dir, f"batch-{len(batch)}-final.json"), "w", encoding="utf-8") as fh:
            json.dump(batch, fh, default=str)

orders.repartition(2).foreachPartition(write_batches)
print("batches written:")
for name in sorted(os.listdir(f"{DATA_DIR}/jdbc_style_batches")):
    print("  ", name)

## 5 · Pub/Sub writes via `foreachPartition`

Same idiom: publish from the executors, batched, with no data through the driver.

In [ ]:
# ------------------------------------------------------------------
# OPTIONAL — Pub/Sub publish via foreachPartition. Uncomment to run.
# ------------------------------------------------------------------
# from google.cloud import pubsub_v1
#
# def publish_partition(iterator, project_id, topic_id):
#     publisher = pubsub_v1.PublisherClient()
#     topic_path = publisher.topic_path(project_id, topic_id)
#     batch = []
#     for row in iterator:
#         batch.append(row.asDict())
#         if len(batch) >= 100:
#             for message in batch:
#                 publisher.publish(topic_path, json.dumps(message, default=str).encode())
#             batch = []
#     for message in batch:
#         publisher.publish(topic_path, json.dumps(message, default=str).encode())
#
# orders.foreachPartition(publish_partition, "your-project-id", "orders-topic")

print("Pub/Sub write cell is commented out — no GCP account required for this notebook.")

## 6 · Fan-out with `cache()`

Writing one DataFrame to three sinks without caching means **three independent
computations** of the same pipeline — three shuffles, three scans.

`cache()` before the fan-out makes the pipeline run once; the three writes then read
from the cache.

**Caution:** Caching is lazy — call `count()` after `cache()` to materialise it, or the first
write will do it implicitly and the other two will reuse it.

In [ ]:
fanout = (
    orders
    .withColumn("amount_with_tax", F.col("amount") * F.lit(1.1))
    .cache()
)
print("materialising the cache with an action:", fanout.count())

# Three sinks, one computation.
fanout.write.mode("overwrite").parquet(f"{DATA_DIR}/sink_parquet")
fanout.write.mode("overwrite").json(f"{DATA_DIR}/sink_json")
fanout.write.mode("overwrite").option("header", True).csv(f"{DATA_DIR}/sink_csv")

print("\nsink row counts:")
print("  parquet:", spark.read.parquet(f"{DATA_DIR}/sink_parquet").count())
print("  json   :", spark.read.json(f"{DATA_DIR}/sink_json").count())
print("  csv    :", spark.read.option("header", True).csv(f"{DATA_DIR}/sink_csv").count())

fanout.unpersist()
print("\ncache released")

## 7 · Idempotency and partition overwrite

A job that re-runs must produce the same result. Two robust strategies:

1. **Partition overwrite** — write to `path/date=...` with `mode("overwrite")`. Re-running
   replaces exactly that partition and leaves the others untouched.
2. **Write to a temp path, then rename** — the write is atomic at the path level; a
   crashed job never leaves a half-written directory visible.

**Caution:** `mode("overwrite")` on a **whole dataset** is not idempotent if other jobs write
concurrently to the same path — it deletes everything, including their output.

In [ ]:
# Partition overwrite: re-running replaces only the partition written.
def write_partitioned_partition(df, path, partition_value, partition_col="order_date"):
    (
        df.filter(F.col(partition_col) == partition_value)
        .write
        .mode("overwrite")
        .partitionBy(partition_col)
        .parquet(path)
    )

idem_path = f"{DATA_DIR}/idempotent_orders"
for run in [1, 2]:
    write_partitioned_partition(orders, idem_path, "2026-10-01")
    total = spark.read.parquet(idem_path).count()
    print(f"run {run}: rows in {idem_path} = {total}   (identical -> idempotent)")

## 8 · Sink checklist

Before a sink ships to production, verify every line:

- [ ] **Write mode** is explicit: `errorifexists` for a new path, `overwrite` for a
      full refresh, `append` for incremental, `ignore` for an idempotent no-op.
- [ ] **Schema** is explicit, and the sink's expected schema matches (`enforceSchema`
      for reads, an explicit `StructType` for the writer).
- [ ] **Partitioning** uses a low-cardinality column (date, region), and each partition
      holds roughly 100 MB or more.
- [ ] **File sizing** is controlled: `repartition` or `coalesce` to avoid tiny files and
      avoid one enormous file.
- [ ] **Compression** is set (`snappy` for Parquet/ORC, `gzip` for CSV).
- [ ] **Idempotency** is established: partition overwrite, or write-to-temp-then-rename.
- [ ] **Staging** is clean: BigQuery has a `temporaryGcsBucket`; temporary paths are
      removed.
- [ ] **Errors** surface: `FAILFAST` on the read side, no silent row drops.
- [ ] **Observability**: row counts logged before and after the write, and a metric on
      the rows written.
- [ ] **Atomicity**: a failed job does not leave a partial dataset visible.
- [ ] **Secrets** come from Secret Manager, never from source code.

## 🔑 Key Takeaways

- Write modes: `append`, `overwrite`, `errorifexists` (safe default), `ignore` (idempotent no-op).
- `partitionBy` prunes on read, but never partition on a high-cardinality column — aim for < ~10,000 partitions and ~100 MB each.
- BigQuery writes require `temporaryGcsBucket`; `writeMethod=direct` streams via the Storage Write API with no staging.
- **JDBC and Pub/Sub writes go through `foreachPartition`** with per-partition batching — create connections inside the function.
- **`cache()` before a fan-out** so the pipeline runs once instead of once per sink.
- Idempotency comes from partition overwrite or write-to-temp-then-rename, not from whole-dataset overwrite.
- A silent overwrite is a data-loss incident — prefer `errorifexists` and explicit modes.

---

## 📚 Next steps

- Time the fan-out with and without `cache()` and compare the task counts in the Spark UI.
- Continue to `07-case-study-retail.ipynb` for a worked end-to-end example.
